# HealthBrowse · DeepSeek 8B on corrected CDC weeks
Import this notebook into Kaggle, enable Internet, select GPU T4 x2, and run all cells. The trainer uses GPU 0. The public training and validation rows are embedded and verified against the corrected hashes. One class-weighted LoRA epoch covers 4,096 records from 2022–2024. Evaluate on the same 256 records from 2025 used for the carry-forward benchmark. The result ZIP includes the adapter, per-record predictions, and validation metrics.

In [ ]:
import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import uuid
import zlib
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select GPU T4 x2 in the Kaggle accelerator settings and restart the session.")
free_gib = torch.cuda.mem_get_info(0)[0] / 2**30
print(f"{torch.cuda.get_device_name(0)} with {free_gib:.1f} GiB available")
if free_gib < 8:
    raise RuntimeError("This recipe requires at least 8 GiB available on GPU 0.")

revision = "7dc835d16da17a48e1ee30d15fb81d7b53f04fc8"
work = Path(tempfile.gettempdir()) / "healthbrowse" / revision
work.mkdir(parents=True, exist_ok=True)
output = Path("/kaggle/working")
output.mkdir(parents=True, exist_ok=True)
env = dict(os.environ, HF_HOME=str(work / "hf-cache"), PYTHONUNBUFFERED="1")
def run(args):
    subprocess.run(args, cwd=work, env=env, check=True)

## Restore the corrected training and validation splits

In [ ]:
def restore(name, encoded, expected_sha, rows):
    raw = zlib.decompress(base64.b64decode(encoded))
    if hashlib.sha256(raw).hexdigest() != expected_sha or len(raw.splitlines()) != rows:
        raise ValueError(f"Incorrect {name} split")
    (work / f"{name}.jsonl").write_bytes(raw)
    print(f"{name}: {rows} records verified")

restore("train", """
...
F7+wtRIoisanGFP1AS92+V71PzpShYo=
""", "ca7208c3d79abecee95e49d31020354e8321422cb86887ebda730a615f9ce21f", 4096)

restore("validation", """
eNrVXUuP2zgSvu+vIHKZBOgOIts9yQR73OsAi7nscVCm2BZjitSSUjuawf73LcruR9J6lGRJZAODTMdW3GWqnl99Vfr7XSm+l+++snf/kk6AE1+Z
y+RBKGWcdB/ZH6IwtpT6wMAK+MpycA54VjlRlvj277//5w9WC7Bf2ebT5u7ywkmII77wkf3bigdpKte8wmzzWSJlHH+P+8q2H9/dsHcK9kJ5AXKR
yip/979//N0uE7DSfJf6trAmrbgXSTieCSt5JoFxoyR770rBP7yW+l4ZK1Pol3c3JO+vP8qLImXt0mYiN6ouJWeVFTn+z9U6tSYXrDCuvE0lWJsJ
UK8FPVSQ90uZDEn56Ucptfnz8Yo/myvaRc6FRikMN5yDYun55RtmSjxg5oQ1B2uqokUhqo/4YimslSWesXBXnvE06W2F1ytoUVeptXCmhCCHmokC
SllKx/gNK1BTNZSgblBX9b20qO+vBQYr/zJ6QNztMuJy0ClaCYPKSnfDHLei0YrXUqZCwQl/DHKzH6RFFfU2hlYEBzSve/EgLEqce89UGH9F1aKq
J+FKpvGljHGhS+sN8CoHNk3+VOhDJc5CMqnvBS+l0f7A/dcQl/dbfIMw9iAhiMx7W/HOmEDThu3SNran2pgWJ4ZnrFDdgxzmA2hu8hpD2K0VeKIl
6JLhn0VWq8YD469DGxRtOjwcIBYS2sslU2lk6kX3itB7xviP8DuZKrCtFZgVmJ8s7QZ9gC4ARfIhui1mpKkSDEpUkOaCAD6ucLIsgXfkYGMEXMjs
/nt2ujcMeFWKiXFss36AsPvKHroCxBE1teLHOsiJomjmLPkl+0J5zxFLqsdocXmnw9xGhLaFEh6Fzgxd8KB3wLuRG10GMS0OeYF+dg/8UdTX8tnM
pIJJNxwiFhLyynoHPw/CZOJUDYgkPnTmYl1JmFRKaiNdoMQBb74vdLodgc9tcvFdchPEAfRiB8aKg9FXntwdHTSQ+gGcfMCqW4sqf1XbglIMDt7N
ooXtYa/iO06qNXkxa2OP0WSG3Sd6tnla7hJh/i3NCYI41uc6B3pVYRYjmzvvUh417Mq6SMnAUrpbc3RWeJ2EVpd1AvRo+lAOnehSuYo1+pKpPHuz
F4UM/oi6665WiN0KtTnP8MtI3ucbiGjdQrrgQKEeXqIXey++c1WlXrinNzAjq4tMPt8Bhvnhy7d9VfnTJR/avPUDpNdCfMnEOzIVmfYmCjoMjvp8
wi0HfD3ONyh1pY/anPScwPozpI4V9KluK6CPoN1QFr9QLarEQT6ZQkzlha+IUGkzw4/Pqvv+3KUoM9AYxvFjystxf4gLRL10KRpvfUC1KFEhHr9E
ZDldGxhRYOqDYVwPYRFYmRoLqQlU3ds9FEKLHP/6onTGuw5auhwdOy/MhxvUpLLNzVGMbrcgVpnXQvmIecN6EMqggLs2D0J596sqof8CBi3Vc086
cs7/UzgOhvnF2kf7c0XF3oOumSsEl8Ixc48ORLB7yKWqLxdp4AKeuqCNf/EOyNsuv1yCaY1RwoJgJkElwz+3v33oz3EcHoGVMXV5L/HtOSi1GDUo
eW+slhBZz8/K+xIrOqVEfX7tOnB3KTzv3NprwHL8SFfg9ymNrZ/8/y26VsMlNL/JYKYMA16WEIPj6ASROgGD+r75sqOzL9qqjh4JAxd5tc+zTmfl
nZh376LIu3NToaxShyEvDFTyJArQZmTVhfdtAg46U/hOVqNP5XhTw3A9nNc5dUYUZQ7oMJuKW6A1p/i3Fm+T+2/kCwHITZg0/2C0aaro19IpU0kn
4eq+6Oc7qjMshC0r19FGpmQigzf2C10NM8AA7m/oOV08J5Di5hnaasHm/U8+L0HvI6aVooOGtCEa9QSqicmkCeIOJ/fA8eWDDJXmcVtj8d548VRe
TUDdzkhA7cG2q2+djDNvYv6/opBhUrXh85wdLNsSrSkHhakhdDBKz2p4acG7txF7yHd7oepub8oK5c0f+wIt3vIMelx1mEkgRHGxkhic8oFG5ueL
oyYRNwVkb8O6sWQKvPLWuyhSpzNkUtTY/6wm+GsPQps2d09lC2xj0BRucjzwkwCFCmPuz5ojrPYJgj9Ymu/drNA5vHyVnjRrnvmOT0RdmM4vJIDL
SRS9h3M+ABarwkA9v+5DPlSonX7cpOuYwwUTnoHm1sg2GAw/C/JAgx0z0AhIMF5Ayiaaf2bmLWVX6JX5iPENtVqEgaV7oIoZE4nNWul3yDTtGi0g
MaDCtPqbYsyz5FIsDq+tt+/IsNVEKlkGJ5AymvuPuSy6XucEG/QDZGtbKnfEAHBpORWVz8ThRXNqCg1yE9mcqjb6dowyLyQ/VujgmVdz4B7bVY/4
6XDZfgrOuVAEG+J4HwT7510jU396Q+sOr8i24Zk8VvpQ6RqGnEdQ2UdPUMQwXudyUKow31t6c9KVVvLSV8TcqCrfSwg/10rhzqLDHh5XGgx+ybQ2
O60bMnZKbbFYR25D/eA+fAjZPzWjrsB/A/nCp57aVQ2r4bUe9OYkmfFGom0mUaRGQXcjjPbGTftnxBxeEq/qzsSs+7L8lpexhx4mdN8qeRR+4BGr
ajeZy7CQD/9LHn/Ki87++RnenDZ3sVkBmqdw0uZBtBOyMk9dSjIGKV6x7dHrJkLCRHPVK3TAYBddEuWzp2u7l+usQOKo1JK3+ZLRxXoSL7Q/Tyf1
Ws7CvTHp3ljdlaTQiEmrj7ZyK3MB58BjXl/RhVTSpnt20ewlC38Lhtkt1GbK6h1LnnkCtejShaqELHzkJGAM9LUo2/VoAd3imqOCzOTwBrYiBF84
M5nTEEkt07swizrgsXrwcLCXnc0s2hxlhDsyaHh0EtUSiqAjlatx5ubhfMW584pKvousHno5e9kOKxfA5f3VjZJk9JADDKwc2ltwxzDBtRTojyoX
YYYytBeR1qaJEvCbgzwekKY0m4VMpiAsNUNNsGUKITjcRHF3zBZ7o2Cufc7bpScI+qjh4Rm1I3FSEioUz+J00px0FItdSAe7jWSmqaiELQ2zwWYa
5h8ZnQnFos65UfaldDvlYES7nphL1okk2FTjrLTVz1tqF62NyOXKj+deKROaiyKDhu0wxOmi7VJYP98iFSOD9/3z7hO5yz6/A8h9NXkAHWQDX+fY
a9i+5FDdErTc6wX35sE0FgpfBwk27dhIgooweKDJXEXLHH3CmHpAPd3CWX3/p5XXbD72BRmXZf0GpkipqNvmzaSHATcAEva7k3hB20hITLFMNPTk
M0H3+SuBhuVq9j5DnRP6l6d8sBW6x/QAnTUaZMBttmO5sKPZ9Lsop7NCrmKd8oCrEWBTPBSTMXF7F+EzeCKZMCNT2IJuIu2ZgyuFL4ScENH7ZPou
20WdMn2MgozcrA70f6u01H1kuQzywmVyqEO/1Jag1mExyrjxNkBJTJonT6LY3ELCk7ZvhtRJAUnWfgbuPADJwmnlNPLEYxnXNpJMWFoazyNJZ4wk
Qba6FxizXa0eINTmyE5YlVQwJ8uvhupbExmYunRdcRS0U7HUKG8wSnbZUN2697HQSKERteVpXY2liZWv+JMLl587or/o3/0+wzD26svXgj795Yfn
JXaD1sMP0U4CZO1BH1Axdf7zVBs08UOQ0xw97E4nCyZRTQLQ8Md1RiIdt6IhQLaLiScfbIaFjHyNwBNCb2TsdmKxrMlejvg+04xkstgqelpus/h6
8v6dB+R+8ib0Y82CUguUaG63tB3rRQPOSBPv9Mj2wy4GLmzU5J2xK7hC6khvZkkjvCVvq0SPZJnYlG2Jl3HZONdrDwyrh9xVS3hWwzBws/5iT5r5
xVfKBx6bnoOyONZJ7N4Sp24hK9vbinfij4Q165u3FUeiTkGunImeY2Tz869X7xAjjZeSHjCwiSPTC7uaIewDjGkw5VK4SaYgrzHNZujLuV/u4m9a
D6pP2wEzaAR3n8mb64c5vEFrydkfRBxyNesCg5dJbHvrqaDq5o0vmFt6IcMlFr5KcqVDyZ3U0Wx+nzYwGA3VmrzKt2eXOYlSsFlkTQIh+s8zg/Fb
Qg0pP3Q1e8xsuK25i8y5kTemb1foF1+7XGAXbauI9OyxyOC/PmofeRZm6ZhycRaTdqmsSOb6BrlwpW+2cawYjB7kdDWwyYilftvVMTUqNpWEbooH
fyhwH5pDmXVcKq+suTq3V9uno8l5bxIl8kLd9TR4umT0pQ+Apj8zfd3Hdr3kADQ5woIHObsLCsYim36YwferD9FbQ4KAPQ9vHfXMqsC5Kp2Kv42E
lofZCaoFoOZCbuDNrbsdwSaLa8HwiLCwXXlmKTSVkGpstPUB2/geObvitqH/AzvBfIM=
""", "e2335b47f738f273956c56bbf636719dc7aee6f461ab1eb23b8075be10da585c", 256)

config = {'source': 'CDC NNDSS x9gk-5huc', 'train_years': [2022, 2023, 2024], 'validation_year': 2025, 'weeks_per_year': 4, 'source_updated_at': 1795030065, 'positive_training_cutoffs': [2.0, 10.0], 'repository_commit': '7dc835d16da17a48e1ee30d15fb81d7b53f04fc8', 'carry_forward_reference': 0.7093946188340807, 'train_sha256': 'ca7208c3d79abecee95e49d31020354e8321422cb86887ebda730a615f9ce21f', 'validation_sha256': 'e2335b47f738f273956c56bbf636719dc7aee6f461ab1eb23b8075be10da585c'}
(work / "split_info.json").write_text(json.dumps(config, indent=2))

## Set up the training environment

In [ ]:
expected = {'prepare.py': 'bda7fdc65665dedcec91ef4467dca66f94a39aaae6775a6d81e4fa409800fd64', 'train.py': 'ec688a0fdf55ed34a04b92d3d510d36ab3e729b105e9e8f0c10d8f9e577bf908'}
root = f"https://raw.githubusercontent.com/abby2395/medical_chatbot/{revision}/healthbrowse/inference/nndss"
for name, digest in expected.items():
    with urllib.request.urlopen(f"{root}/{name}", timeout=60) as response:
        script = response.read()
    if hashlib.sha256(script).hexdigest() != digest:
        raise ValueError(f"Unexpected source for {name}")
    (work / name).write_bytes(script)
venv = work / "env"
python = str(venv / "bin" / "python")
if not Path(python).is_file():
    run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(venv)])
run([python, "-m", "pip", "install", "--no-cache-dir", "--disable-pip-version-check",
     "transformers==4.57.6", "peft==0.17.1", "accelerate==1.10.1", "bitsandbytes==0.48.1"])
run([python, "-c", "from train import require_gpu; require_gpu('deepseek-ai/DeepSeek-R1-Distill-Llama-8B')"])

## Train one epoch and compare with carry-forward
The training seed, model revision, optimizer settings, and class weights follow the earlier 4,096-row run. The count bands use the original cutoffs of 2 and 10. The carry-forward baseline scores a missing previous count as zero. Inspect the per-class scores alongside macro F1.

In [ ]:
(work / 'full_run.py').write_text(r'''
"""One class-weighted LoRA epoch and a fixed 2025 validation comparison."""
import csv
import hashlib
import json
import random
import sys
import time
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from transformers import AutoTokenizer, set_seed
from train import CHECKPOINTS, LABELS, SEED, attach_adapter, encode, load_model, require_gpu, scores

MODEL = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
BATCH = 32
MICRO = 8
LR = 2e-4

def load_rows(name, sha):
    raw = Path(name + ".jsonl").read_bytes()
    if hashlib.sha256(raw).hexdigest() != sha:
        raise ValueError(f"Unexpected fingerprint for {name}")
    return [json.loads(line) for line in raw.splitlines()]

def pack(examples, pad, device):
    width = max(len(ids) for ids, _ in examples)
    ids = torch.tensor([[pad] * (width-len(seq)) + seq for seq, _ in examples], device=device)
    lengths = torch.tensor([len(seq) for seq, _ in examples], device=device)
    mask = torch.arange(width,device=device)[None,:] >= width - lengths[:,None]
    pos = (mask.long().cumsum(dim=1) - 1).clamp_min(0)
    targets = torch.tensor([label for _, label in examples], device=device)
    return ids, mask.long(), pos, targets

def choice_logits(model, packed, choices):
    ids, mask, positions, targets = packed
    with torch.autocast(device_type=ids.device.type, dtype=torch.float16, enabled=ids.is_cuda):
        output = model(input_ids=ids, attention_mask=mask, position_ids=positions,
                       use_cache=False, logits_to_keep=1)
    return output.logits[:, -1, choices].float(), targets

def train_epoch(model, examples, choices, pad):
    model.train()
    device = model.get_input_embeddings().weight.device
    counts = Counter(k for _,k in examples)
    weights = torch.tensor([len(examples)/(len(LABELS)*counts[k]) for k in range(len(LABELS))],device=device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, foreach=False)
    order = list(range(len(examples)))
    random.Random(SEED).shuffle(order)
    started = time.monotonic()
    seen = steps = 0
    for start in range(0,len(order),BATCH):
        indices = order[start:start+BATCH]
        opt.zero_grad(set_to_none=True)
        for offset in range(0,len(indices),MICRO):
            group = [examples[i] for i in indices[offset:offset+MICRO]]
            logits, targets = choice_logits(model, pack(group,pad,device),choices)
            # A fixed batch denominator preserves the inverse-frequency class weighting.
            loss = F.cross_entropy(logits,targets,weight=weights,reduction="sum") / len(indices)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite training loss")
            loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1., error_if_nonfinite=True)
        opt.step()
        seen += len(indices)
        steps += 1
        if steps == 1 or steps % 8 == 0 or seen == len(order):
            print(f"Trained {seen}/{len(order)} rows; step {steps}; elapsed {time.monotonic()-started:.0f}s",flush=True)
    return steps

@torch.inference_mode()
def predict(model, examples, choices, pad):
    model.eval()
    device = model.get_input_embeddings().weight.device
    decisions=[]
    for i in range(0,len(examples),MICRO):
        logits,_=choice_logits(model,pack(examples[i:i+MICRO],pad,device),choices)
        decisions.extend(LABELS[k] for k in logits.argmax(1).tolist())
    return decisions

def describe(y,p):
    return dict(scores(y,p),confusion_matrix=confusion_matrix(y,p,labels=LABELS).tolist(),
                predicted_counts=dict(Counter(p)),positive_exact=sum(a==b and a!=LABELS[0] for a,b in zip(y,p)),
                positive_detected=sum(a!=LABELS[0] and b!=LABELS[0] for a,b in zip(y,p)),
                false_positive_count=sum(a==LABELS[0] and b!=LABELS[0] for a,b in zip(y,p)))

def main():
    out = Path(sys.argv[1])
    require_gpu(MODEL)
    set_seed(SEED)
    config = json.loads(Path("split_info.json").read_text())
    train = load_rows("train",config["train_sha256"])
    val = load_rows("validation",config["validation_sha256"])
    assert len(train)==4096 and len(val)==256 and not {r["text"] for r in train}.intersection(r["text"] for r in val)
    z = [r["label"] for r in train]
    y = [r["label"] for r in val]
    lr = make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=500,class_weight="balanced",random_state=SEED))
    lr.fit([r["text"] for r in train],z)
    reference = lr.predict([r["text"] for r in val]).tolist()
    low, medium = config["positive_training_cutoffs"]
    carry = []
    for row in val:
        prior = row["text"].rsplit("Previous week reported cases: ", 1)[1].removesuffix(".")
        count = 0 if prior == "unknown" else int(prior)
        band = 0 if count == 0 else 1 if count <= low else 2 if count <= medium else 3
        carry.append(LABELS[band])
    carry_metrics = describe(y, carry)
    if abs(carry_metrics["macro_f1"] - config["carry_forward_reference"]) > 1e-10:
        raise ValueError("The carry-forward comparison did not reproduce on the restored validation set")
    print("Validation carry-forward macro F1", carry_metrics["macro_f1"], flush=True)
    model = attach_adapter(load_model(MODEL))
    tokenizer = AutoTokenizer.from_pretrained(MODEL,revision=CHECKPOINTS[MODEL])
    examples,choices = encode(tokenizer,train)
    validation,other_choices = encode(tokenizer,val)
    assert choices==other_choices
    steps=train_epoch(model,examples,choices,tokenizer.eos_token_id)
    model.save_pretrained(out/"adapter",safe_serialization=True,save_embedding_layers=False)
    predictions=predict(model,validation,choices,tokenizer.eos_token_id)
    metrics=dict(config,model=MODEL,model_revision=CHECKPOINTS[MODEL],labels=LABELS,seed=SEED,
        epochs=1,batch_size=BATCH,micro_batch_size=MICRO,optimizer_steps=steps,learning_rate=LR,
        training_rows=len(train),training_support=dict(Counter(z)),validation_rows=len(val),
        validation_support=dict(Counter(y)),class_weights={k:len(z)/(len(LABELS)*z.count(k)) for k in LABELS},
        adapter=describe(y,predictions),carry_forward=carry_metrics,logistic_regression=describe(y,reference),
        majority=describe(y,[Counter(z).most_common(1)[0][0]]*len(y)))
    with (out/"validation_predictions.csv").open("w",newline="") as f:
        w=csv.writer(f)
        w.writerow(["text","actual","deepseek_adapter","logistic_regression","carry_forward"])
        w.writerows((r["text"],r["label"],p,q,c) for r,p,q,c in zip(val,predictions,reference,carry,strict=True))
    (out/"validation_metrics.json").write_text(json.dumps(metrics,indent=2)+"\n")
    for name in ("adapter","carry_forward","logistic_regression","majority"):
        print(name,json.dumps(metrics[name]),flush=True)

if __name__=="__main__":
    main()
''')
run_id = "healthbrowse-deepseek-8b-corrected-" + uuid.uuid4().hex[:8]
result = work / run_id
result.mkdir()
run([python, "full_run.py", str(result)])
archive = shutil.make_archive(str(output / run_id), "zip", root_dir=result)
metrics = json.loads((result / "validation_metrics.json").read_text())
for name in ("adapter", "carry_forward", "logistic_regression", "majority"):
    print(f"{name:20s} macro F1 = {metrics[name]['macro_f1']:.4f}")
print(f"Download {archive} from the Kaggle output files.")